# 02 — Tratamiento Bronze → Silver de telemetría

## Objetivo

Convertir el archivo original de telemetría en dos salidas:

- **Silver:** lecturas utilizables y con formato consistente.
- **Cuarentena:** lecturas que no pueden corregirse de manera segura y necesitan revisión.

La unidad de observación continúa siendo **una lectura de telemetría de un viaje en un momento determinado**.

Este notebook evita inventar mediciones: no utiliza media, mediana, moda, `ffill` ni `bfill` para completar temperatura o humedad.

## 1. Librerías, rutas y reglas

In [1]:
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 100)

CONFIG = {
    "ruta_bronze": Path(
        r"C:/Users/remrodri/Github/practicasNotebookColab/"
        r"proyecto-integradorV2/datos/bronze/andinalog_iot_telemetry.csv"
    ),
    "carpeta_salida": Path("outputs/02_bronze_silver_iot"),
    "zona_horaria_origen": "America/La_Paz",
    "zona_horaria_salida": "UTC",
    "unidad_canonica": "C",
    "centinelas_temperatura": [-999.0],
    "rango_humedad": (0.0, 100.0),
    "columnas_obligatorias": [
        "timestamp", "viaje_id", "order_id", "camion_id", "producto_id",
        "temperatura_cabina_c", "temp_unit", "humedad_cabina_pct",
        "desviacion_termica_flag", "desviacion_proximos_60min_flag",
    ],
    "patrones_id": {
        "viaje_id": r"VIA-\d{5}",
        "order_id": r"ORD-\d{4}-\d{5}",
        "camion_id": r"CAM-\d{2}",
        "producto_id": r"PROD-\d{3}",
    },
}

CONFIG["carpeta_salida"].mkdir(parents=True, exist_ok=True)
RUTA_SILVER = CONFIG["carpeta_salida"] / "andinalog_iot_telemetry_silver.csv"
RUTA_CUARENTENA = CONFIG["carpeta_salida"] / "andinalog_iot_telemetry_quarantine.csv"
RUTA_INFORME = CONFIG["carpeta_salida"] / "Informe_02_Bronze_Silver_IoT.md"

print("Reglas principales:")
print("- Temperatura final en Celsius.")
print("- Fechas locales de Bolivia convertidas a UTC.")
print("- Humedad válida entre 0% y 100%.")
print("- No se imputan mediciones ausentes.")

Reglas principales:
- Temperatura final en Celsius.
- Fechas locales de Bolivia convertidas a UTC.
- Humedad válida entre 0% y 100%.
- No se imputan mediciones ausentes.


### Explicación sencilla

La configuración reúne todas las reglas en un solo lugar. Así se puede revisar fácilmente qué considera válido el notebook y dónde guardará los resultados.

## 2. Carga del archivo original

In [2]:
ruta_bronze = CONFIG["ruta_bronze"]
if not ruta_bronze.exists():
    alternativa = Path("/content/andinalog_iot_telemetry.csv")
    if alternativa.exists():
        ruta_bronze = alternativa
    else:
        raise FileNotFoundError("No se encontró el CSV Bronze. Ajusta CONFIG['ruta_bronze'].")

bronze = pd.read_csv(ruta_bronze, dtype=str, keep_default_na=False)
faltan_columnas = [c for c in CONFIG["columnas_obligatorias"] if c not in bronze.columns]
if faltan_columnas:
    raise ValueError(f"Faltan columnas obligatorias: {faltan_columnas}")

print("Archivo:", ruta_bronze)
print(f"Bronze: {len(bronze):,} filas y {bronze.shape[1]} columnas")
display(bronze.head(3))

Archivo: C:\Users\remrodri\Github\practicasNotebookColab\proyecto-integradorV2\datos\bronze\andinalog_iot_telemetry.csv
Bronze: 28,920 filas y 10 columnas


,timestamp,viaje_id,order_id,camion_id,producto_id,temperatura_cabina_c,temp_unit,humedad_cabina_pct,desviacion_termica_flag,desviacion_proximos_60min_flag
0,2026-08-11 14:11:00,VIA-00001,ORD-2026-05710,CAM-20,PROD-045,2.19,C,72.6,0,0
1,2026-08-11 14:41:00,VIA-00001,ORD-2026-05710,CAM-20,PROD-045,4.91,C,75.0,0,0
2,2026-08-11 15:11:00,VIA-00001,ORD-2026-05710,CAM-20,PROD-045,2.76,C,94.7,0,0


### Explicación sencilla

El archivo se lee como texto para conservar exactamente lo que llegó desde el origen. Las conversiones se harán sobre una copia y los valores originales quedarán disponibles para auditoría.

## 3. Funciones de tratamiento

In [3]:
def preparar_auditoria(df):
    '''Crea una copia de trabajo y conserva el número de fila original.'''
    salida = df.copy()
    salida.insert(0, "_fila_bronze", np.arange(1, len(salida) + 1))
    salida["timestamp_original"] = salida["timestamp"]
    salida["camion_id_original"] = salida["camion_id"]
    salida["temperatura_original"] = salida["temperatura_cabina_c"]
    salida["temp_unit_original"] = salida["temp_unit"]
    salida["humedad_original"] = salida["humedad_cabina_pct"]
    salida["fue_transformada"] = False
    salida["fue_imputada"] = False
    salida["metodo_imputacion"] = "no_aplica"
    salida["motivo_imputacion"] = "no_se_imputan_mediciones_ambiguas"
    return salida


def normalizar_identificadores(df):
    '''Corrige únicamente espacios y uso de mayúsculas.'''
    salida = df.copy()
    for columna in CONFIG["patrones_id"]:
        normalizado = salida[columna].str.strip().str.upper()
        cambio = normalizado.ne(salida[columna])
        salida[columna] = normalizado
        salida["fue_transformada"] |= cambio
    return salida


def convertir_timestamp(df):
    '''Interpreta hora boliviana y la expresa en UTC; conserva el texto original.'''
    salida = df.copy()
    texto = salida["timestamp_original"].str.strip()
    fecha_local = pd.to_datetime(texto, errors="coerce")
    fecha_utc = fecha_local.dt.tz_localize(
        CONFIG["zona_horaria_origen"], ambiguous="NaT", nonexistent="NaT"
    ).dt.tz_convert(CONFIG["zona_horaria_salida"])
    salida["timestamp_invalido"] = fecha_utc.isna()
    salida["timestamp"] = fecha_utc
    salida["fue_transformada"] |= fecha_utc.notna()
    return salida


def convertir_temperatura(df):
    '''Convierte Fahrenheit a Celsius y reconoce ausencias sin inventar valores.'''
    salida = df.copy()
    texto = salida["temperatura_original"].str.strip()
    numero = pd.to_numeric(texto.replace("", np.nan), errors="coerce")
    unidad = salida["temp_unit_original"].str.strip().str.upper()

    salida["temperatura_vacia"] = texto.eq("")
    salida["temperatura_no_numerica"] = texto.ne("") & numero.isna()
    salida["temperatura_centinela"] = numero.isin(CONFIG["centinelas_temperatura"])
    salida["unidad_no_admitida"] = ~unidad.isin(["C", "F"])

    convertida = numero.copy()
    es_fahrenheit = unidad.eq("F") & numero.notna()
    convertida.loc[es_fahrenheit] = (numero.loc[es_fahrenheit] - 32) * 5 / 9
    convertida.loc[salida["temperatura_centinela"] | salida["unidad_no_admitida"]] = np.nan

    salida["temperatura_cabina_c"] = convertida.round(4)
    salida["temp_unit"] = "C"
    salida["temperatura_convertida_fahrenheit"] = es_fahrenheit
    salida["fue_transformada"] |= es_fahrenheit | salida["temperatura_centinela"]
    return salida


def convertir_humedad_y_flags(df):
    '''Convierte humedad e indicadores a tipos numéricos.'''
    salida = df.copy()
    texto_h = salida["humedad_original"].str.strip()
    humedad = pd.to_numeric(texto_h.replace("", np.nan), errors="coerce")
    salida["humedad_vacia"] = texto_h.eq("")
    salida["humedad_no_numerica"] = texto_h.ne("") & humedad.isna()
    salida["humedad_fuera_rango"] = humedad.notna() & ~humedad.between(*CONFIG["rango_humedad"])
    salida["humedad_cabina_pct"] = humedad

    for columna in ["desviacion_termica_flag", "desviacion_proximos_60min_flag"]:
        original = salida[columna].str.strip()
        numerico = pd.to_numeric(original.replace("", np.nan), errors="coerce")
        salida[f"{columna}_invalido"] = ~numerico.isin([0, 1])
        salida[columna] = numerico.astype("Int64")
    return salida

In [4]:
def clasificar_duplicados(df):
    '''Separa copias redundantes de lecturas con contenido contradictorio.'''
    salida = df.copy()
    clave = ["viaje_id", "timestamp_original"]
    contenido = [
        "order_id", "camion_id", "producto_id", "temperatura_cabina_c", "temp_unit",
        "humedad_cabina_pct", "desviacion_termica_flag", "desviacion_proximos_60min_flag",
    ]

    firma = salida[contenido].apply(
        lambda fila: "|".join(map(str, fila.tolist())), axis=1
    )
    diversidad = firma.groupby([salida[c] for c in clave]).transform("nunique")
    tamano_grupo = salida.groupby(clave, dropna=False)["_fila_bronze"].transform("size")

    salida["duplicado_conflictivo"] = tamano_grupo.gt(1) & diversidad.gt(1)
    salida["duplicado_redundante"] = (
        tamano_grupo.gt(1)
        & diversidad.eq(1)
        & salida.duplicated(clave, keep="first")
    )
    return salida


def validar_y_clasificar(df):
    '''Calcula los errores residuales y decide el destino de cada fila.'''
    salida = df.copy()

    for columna, patron in CONFIG["patrones_id"].items():
        salida[f"{columna}_invalido"] = ~salida[columna].str.fullmatch(patron)

    reglas_error = {
        "timestamp_invalido": "fecha_invalida",
        "temperatura_vacia": "temperatura_vacia",
        "temperatura_no_numerica": "temperatura_no_numerica",
        "temperatura_centinela": "temperatura_centinela",
        "unidad_no_admitida": "unidad_no_admitida",
        "humedad_vacia": "humedad_vacia",
        "humedad_no_numerica": "humedad_no_numerica",
        "humedad_fuera_rango": "humedad_fuera_rango",
        "desviacion_termica_flag_invalido": "flag_actual_invalido",
        "desviacion_proximos_60min_flag_invalido": "flag_futuro_invalido",
        "viaje_id_invalido": "viaje_id_invalido",
        "order_id_invalido": "order_id_invalido",
        "camion_id_invalido": "camion_id_invalido",
        "producto_id_invalido": "producto_id_invalido",
        "duplicado_redundante": "duplicado_redundante",
        "duplicado_conflictivo": "duplicado_conflictivo",
    }

    def motivos_fila(fila):
        return "; ".join(motivo for bandera, motivo in reglas_error.items() if bool(fila[bandera]))

    salida["calidad_motivo"] = salida.apply(motivos_fila, axis=1)
    salida["cantidad_errores"] = salida[list(reglas_error)].sum(axis=1).astype(int)
    salida["calidad_estado"] = np.where(salida["cantidad_errores"].eq(0), "SILVER", "CUARENTENA")

    columnas_originales = [
        "timestamp_original", "camion_id_original", "temperatura_original",
        "temp_unit_original", "humedad_original",
    ]
    salida["cantidad_transformaciones"] = (
        salida["fue_transformada"].astype(int)
        + salida["temperatura_convertida_fahrenheit"].astype(int)
        + salida["camion_id"].ne(salida["camion_id_original"]).astype(int)
    )
    return salida

### Explicación sencilla

Las funciones separan el proceso en pasos pequeños: conservar originales, normalizar códigos, convertir fechas, convertir mediciones y decidir el destino final. Esto hace más fácil revisar por qué cambió cada dato.

## 4. Ejecución del tratamiento

In [5]:
tratado = (
    bronze
    .pipe(preparar_auditoria)
    .pipe(normalizar_identificadores)
    .pipe(convertir_timestamp)
    .pipe(convertir_temperatura)
    .pipe(convertir_humedad_y_flags)
    .pipe(clasificar_duplicados)
    .pipe(validar_y_clasificar)
)

print(f"Filas procesadas: {len(tratado):,}")
display(tratado[[
    "_fila_bronze", "timestamp_original", "timestamp", "camion_id_original", "camion_id",
    "temperatura_original", "temp_unit_original", "temperatura_cabina_c",
    "humedad_original", "humedad_cabina_pct", "calidad_estado", "calidad_motivo",
]].head())

Filas procesadas: 28,920


,_fila_bronze,timestamp_original,timestamp,camion_id_original,camion_id,temperatura_original,temp_unit_original,temperatura_cabina_c,humedad_original,humedad_cabina_pct,calidad_estado,calidad_motivo
0,1,2026-08-11 14:11:00,2026-08-11 18:11:00+00:00,CAM-20,CAM-20,2.19,C,2.19,72.6,72.6,SILVER,
1,2,2026-08-11 14:41:00,2026-08-11 18:41:00+00:00,CAM-20,CAM-20,4.91,C,4.91,75.0,75.0,SILVER,
2,3,2026-08-11 15:11:00,2026-08-11 19:11:00+00:00,CAM-20,CAM-20,2.76,C,2.76,94.7,94.7,SILVER,
3,4,2026-08-11 15:41:00,2026-08-11 19:41:00+00:00,CAM-20,CAM-20,3.96,C,3.96,74.0,74.0,SILVER,
4,5,2026-08-11 16:11:00,2026-08-11 20:11:00+00:00,CAM-20,CAM-20,4.49,C,4.49,70.3,70.3,SILVER,


### Explicación sencilla

La tabla muestra el valor recibido junto al valor tratado. Por ejemplo, permite comprobar el cambio de un código como `cam-20 ` a `CAM-20`, o una temperatura Fahrenheit a su equivalente en Celsius.

## 5. Resultados de las reglas

In [6]:
resumen_reglas = pd.DataFrame({
    "regla": [
        "Códigos de camión normalizados", "Temperaturas Fahrenheit convertidas",
        "Temperaturas centinela -999", "Unidades no admitidas",
        "Temperaturas vacías", "Humedades vacías", "Humedades fuera de rango",
        "Fechas inválidas", "Duplicados redundantes", "Filas en grupos duplicados conflictivos",
    ],
    "filas": [
        int(tratado["camion_id"].ne(tratado["camion_id_original"]).sum()),
        int(tratado["temperatura_convertida_fahrenheit"].sum()),
        int(tratado["temperatura_centinela"].sum()),
        int(tratado["unidad_no_admitida"].sum()),
        int(tratado["temperatura_vacia"].sum()),
        int(tratado["humedad_vacia"].sum()),
        int(tratado["humedad_fuera_rango"].sum()),
        int(tratado["timestamp_invalido"].sum()),
        int(tratado["duplicado_redundante"].sum()),
        int(tratado["duplicado_conflictivo"].sum()),
    ],
})
display(resumen_reglas)

,regla,filas
0,Códigos de camión normalizados,50
1,Temperaturas Fahrenheit convertidas,50
2,Temperaturas centinela -999,120
3,Unidades no admitidas,5
4,Temperaturas vacías,80
5,Humedades vacías,100
6,Humedades fuera de rango,15
7,Fechas inválidas,15
8,Duplicados redundantes,115
9,Filas en grupos duplicados conflictivos,10


In [7]:
print("Interpretación sencilla")
print("- Las normalizaciones de escritura y las conversiones Fahrenheit → Celsius son correcciones seguras.")
print("- Una excursión térmica real no es un error de calidad; se conserva.")
print("- Las mediciones ausentes, fechas imposibles, Kelvin y conflictos no se adivinan: pasan a cuarentena.")
print("- No se realizó ninguna imputación de temperatura ni humedad.")

Interpretación sencilla
- Las normalizaciones de escritura y las conversiones Fahrenheit → Celsius son correcciones seguras.
- Una excursión térmica real no es un error de calidad; se conserva.
- Las mediciones ausentes, fechas imposibles, Kelvin y conflictos no se adivinan: pasan a cuarentena.
- No se realizó ninguna imputación de temperatura ni humedad.


## 6. Separación entre Silver y cuarentena

In [8]:
silver = tratado.loc[tratado["calidad_estado"].eq("SILVER")].copy()
cuarentena = tratado.loc[tratado["calidad_estado"].eq("CUARENTENA")].copy()

print(f"Silver: {len(silver):,} filas")
print(f"Cuarentena: {len(cuarentena):,} filas")
print(f"Total conciliado: {len(silver) + len(cuarentena):,} de {len(bronze):,} filas Bronze")

print("\nMotivos de cuarentena:")
motivos = (
    cuarentena["calidad_motivo"].str.split("; ").explode()
    .value_counts().rename_axis("motivo").reset_index(name="filas")
)
display(motivos)

Silver: 28,465 filas
Cuarentena: 455 filas
Total conciliado: 28,920 de 28,920 filas Bronze

Motivos de cuarentena:


,motivo,filas
0,temperatura_centinela,120
1,duplicado_redundante,115
2,humedad_vacia,100
3,temperatura_vacia,80
4,humedad_fuera_rango,15
5,fecha_invalida,15
6,duplicado_conflictivo,10
7,unidad_no_admitida,5


### Explicación sencilla

Silver contiene únicamente lecturas que quedaron utilizables después de aplicar correcciones seguras. Cuarentena conserva todas las filas que necesitan una decisión humana o una fuente adicional. No se eliminó ninguna fila silenciosamente.

## 7. Controles finales

In [9]:
controles = pd.DataFrame([
    {
        "control": "Bronze = Silver + cuarentena",
        "resultado": len(bronze) == len(silver) + len(cuarentena),
        "detalle": f"{len(bronze)} = {len(silver)} + {len(cuarentena)}",
    },
    {
        "control": "Silver sin errores residuales",
        "resultado": bool(silver["cantidad_errores"].eq(0).all()),
        "detalle": f"errores máximos por fila: {int(silver['cantidad_errores'].max()) if len(silver) else 0}",
    },
    {
        "control": "Silver en Celsius",
        "resultado": bool(silver["temp_unit"].eq("C").all()),
        "detalle": str(silver["temp_unit"].value_counts().to_dict()),
    },
    {
        "control": "Silver con humedad válida",
        "resultado": bool(silver["humedad_cabina_pct"].between(0, 100).all()),
        "detalle": "rango 0 a 100",
    },
    {
        "control": "Silver sin clave viaje + timestamp repetida",
        "resultado": not silver.duplicated(["viaje_id", "timestamp"]).any(),
        "detalle": f"duplicados: {int(silver.duplicated(['viaje_id', 'timestamp']).sum())}",
    },
    {
        "control": "No se imputaron mediciones",
        "resultado": not bool(tratado["fue_imputada"].any()),
        "detalle": f"filas imputadas: {int(tratado['fue_imputada'].sum())}",
    },
])
display(controles)

if not controles["resultado"].all():
    raise AssertionError("Uno o más controles finales no se cumplieron.")
print("Todos los controles finales se cumplieron.")

,control,resultado,detalle
0,Bronze = Silver + cuarentena,True,28920 = 28465 + 455
1,Silver sin errores residuales,True,errores máximos por fila: 0
2,Silver en Celsius,True,{'C': 28465}
3,Silver con humedad válida,True,rango 0 a 100
4,Silver sin clave viaje + timestamp repetida,True,duplicados: 0
5,No se imputaron mediciones,True,filas imputadas: 0


Todos los controles finales se cumplieron.


## 8. Exportación de resultados

In [10]:
# Los timestamps se escriben como texto ISO en UTC para que sean portables.
for tabla in [silver, cuarentena]:
    tabla["timestamp"] = tabla["timestamp"].astype("string")

silver.to_csv(RUTA_SILVER, index=False, encoding="utf-8-sig")
cuarentena.to_csv(RUTA_CUARENTENA, index=False, encoding="utf-8-sig")

print("Archivos CSV generados:")
print("-", RUTA_SILVER.resolve())
print("-", RUTA_CUARENTENA.resolve())

Archivos CSV generados:
- C:\Users\remrodri\Documents\Codex\2026-09-28\referenced-chatgpt-conversation-this-is-an\outputs\02_bronze_silver_iot\andinalog_iot_telemetry_silver.csv
- C:\Users\remrodri\Documents\Codex\2026-09-28\referenced-chatgpt-conversation-this-is-an\outputs\02_bronze_silver_iot\andinalog_iot_telemetry_quarantine.csv


## 9. Informe automático de la ejecución

In [11]:
fecha_ejecucion = datetime.now(timezone.utc).isoformat()
motivos_texto = "\n".join(
    f"- {fila.motivo}: {int(fila.filas):,} filas" for fila in motivos.itertuples()
)

informe = f'''# Informe 02 — Tratamiento Bronze a Silver de IoT

## Objetivo

Preparar las lecturas de telemetría de AndinaLog sin inventar mediciones y separar los errores no resolubles en cuarentena.

## Unidad de observación

Una fila representa una lectura de telemetría de un viaje en un momento determinado.

## Resultados

- Filas Bronze: {len(bronze):,}
- Filas Silver: {len(silver):,}
- Filas en cuarentena: {len(cuarentena):,}
- Códigos de camión normalizados: {int(tratado['camion_id'].ne(tratado['camion_id_original']).sum()):,}
- Temperaturas Fahrenheit convertidas: {int(tratado['temperatura_convertida_fahrenheit'].sum()):,}
- Filas imputadas: {int(tratado['fue_imputada'].sum()):,}

## Motivos de cuarentena

{motivos_texto}

## Decisiones principales

- Celsius es la unidad final.
- Fahrenheit se convierte mediante una fórmula exacta.
- Kelvin permanece en cuarentena porque no es una unidad esperada.
- `-999` se reconoce como ausencia de medición.
- Las fechas imposibles no se adivinan.
- Los duplicados redundantes y conflictivos se conservan en cuarentena con su motivo.
- No se imputaron temperaturas ni humedades.

## Conciliación

{len(bronze):,} filas Bronze = {len(silver):,} filas Silver + {len(cuarentena):,} filas en cuarentena.

## Limitaciones

No se validó la existencia de viajes, camiones, órdenes o productos contra tablas maestras porque esas fuentes no forman parte de este tratamiento. Esta ausencia de enriquecimiento no invalida por sí sola una lectura correcta.

Ejecución UTC: {fecha_ejecucion}
'''

RUTA_INFORME.write_text(informe, encoding="utf-8")
print("Informe generado:", RUTA_INFORME.resolve())
print("\nResumen final:")
print(f"Bronze {len(bronze):,} → Silver {len(silver):,} + Cuarentena {len(cuarentena):,}")

Informe generado: C:\Users\remrodri\Documents\Codex\2026-09-28\referenced-chatgpt-conversation-this-is-an\outputs\02_bronze_silver_iot\Informe_02_Bronze_Silver_IoT.md

Resumen final:
Bronze 28,920 → Silver 28,465 + Cuarentena 455


## 10. Conclusión sencilla

El archivo Silver contiene las lecturas que pueden utilizarse con confianza después de aplicar únicamente correcciones claras. La cuarentena no es basura: conserva los registros cuyo problema no puede resolverse sin inventar información.

El siguiente notebook construirá la tabla **Gold**, donde se crearán las variables temporales necesarias para el modelo, como la temperatura anterior y la variación durante 30 minutos.